## Libraries

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier



: 

## EDA

In [ ]:
df = pd.read_excel('online_course_recommendation.xlsx')

In [ ]:
df.shape

In [ ]:
df.index

### Columns and Data Types

In [ ]:
df.columns

In [ ]:
numerical_columns=df.select_dtypes(include='number').columns
numerical_columns

In [ ]:
df.select_dtypes(include='number').columns.value_counts().sum()


In [ ]:
categorical_columns=df.select_dtypes(include=['object', 'category']).columns
categorical_columns


In [ ]:
df.select_dtypes(include=['object', 'category']).columns.value_counts().sum()


In [ ]:
df.dtypes

In [ ]:
df.info()

Based on the rough analysis on the data we see <mark> no null value </mark> 




### descriptive statistics

Descriptive Statistics for Numerical columns


In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum()

In [ ]:
df.shape

In [ ]:
for col in df.columns:
    print('==' * 50)

    print(f"--- Column: {col} ---")
    
    print(df[col].value_counts())  # Shows the first 10 unique values


### Identifying and Handeling Duplicates

In [ ]:
df.duplicated()

In [ ]:
print(df.duplicated().sum())

### Outliers    


In [ ]:
for col in numerical_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR
    
    # Count outliers for this specific column
    num_outliers = df[(df[col] < lower_limit) | (df[col] > upper_limit)].shape[0]
    
    print(f"Column: {col:20} | Outliers: {num_outliers:5} | Limits: ({lower_limit:.2f}, {upper_limit:.2f})")

In [ ]:
# Loop through each numeric column and print the plot
for col in numerical_columns:
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=df[col], color='salmon')
    plt.title(f'Outlier Detection: {col}')
    plt.show()


### Visualizatoins

In [ ]:
corr_matrix = df[numerical_columns].corr()

plt.figure(figsize=(10, 8))

# Create a mask to hide the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, 
    annot=True, 
    cmap='coolwarm', 
    fmt=".5f", 
    linewidths=0.5, 
    vmin=-1, vmax=1
)

plt.title('Triangular Correlation Heatmap', fontsize=16, pad=15)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x='rating', y='feedback_score')
plt.title('Checking Relationship')
plt.show()


In [ ]:
cols = 3
all_cols = df.columns.tolist()
n = len(all_cols)
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 4))
axes = axes.flatten()

for ax, col in zip(axes, all_cols):
    if pd.api.types.is_numeric_dtype(df[col]):
        sns.histplot(
            df[col].dropna(),
            kde=True,
            stat="density",
            bins=20,
            color="skyblue",
            edgecolor="black",
            ax=ax,
        )
        ax.set_ylabel("Density")
        ax.set_title(f"{col} — Numeric")
    else:
        counts = df[col].value_counts()
        sns.barplot(
            x=counts.index.astype(str),
            y=counts.values,
            palette="pastel",
            ax=ax,
        )
        x = np.arange(len(counts))
        if len(x) > 1:
            x_smooth = np.linspace(x.min(), x.max(), 200)
            y_smooth = np.interp(x_smooth, x, counts.values)
            ax2 = ax.twinx()
            ax2.plot(x_smooth, y_smooth, color="red", alpha=0.85)
            ax2.set_ylabel("Smoothed count", color="red")
            ax2.tick_params(axis="y", labelcolor="red")
        ax.set_xticklabels(counts.index.astype(str), rotation=45, ha="right")
        ax.set_ylabel("Count")
        ax.set_title(f"{col} — Categorical")

    ax.set_xlabel(col)

for extra_ax in axes[n:]:
    fig.delaxes(extra_ax)

plt.tight_layout()
plt.show()

## Creating new features

In [ ]:
def calculate_engagement_index(df):

    time_taken = df['time_spent_hours']
    duration = df['course_duration_hours']
    sweet_spot_limit = 1.3 * duration

    # Define our 3 conditions
    conditions = [
        (time_taken < duration),                                      # 1. Under-Time
        (time_taken >= duration) & (time_taken <= sweet_spot_limit), # 2. Sweet-Spot
        (time_taken > sweet_spot_limit)                              # 3. Over-Time
    ]

    # Define the mathematical formulas for each condition
    under_time_formula = time_taken / duration
    sweet_spot_formula = 1.0
    over_time_formula = 1.0 - ((time_taken - sweet_spot_limit) / duration)

    # Apply the over-time formula but enforce the 0.1 lower bound floor using np.maximum
    over_time_penalized = np.maximum(over_time_formula, 0.1)

    # Choices array matching our conditions
    choices = [under_time_formula, sweet_spot_formula, over_time_penalized]

    # Execute vectorization
    df['Engagement_Index'] = np.select(conditions, choices, default=0.0)
    
    return df

# Apply the function to your dataframe
df = calculate_engagement_index(df)

# Let's check a quick statistical summary of your new engineered feature!
print(df['Engagement_Index'].describe())
print("=="*50)
print()
# Sample one user from each behavior bracket to verify the logic
print(df[['course_duration_hours', 'time_spent_hours', 'Engagement_Index']].head(10))

Category 1: Linear Ramping (Under-Time Learners)When it's used: When a user's spent time is strictly less than the course duration <mark> (time_spent_hours < course_duration_hours).The Calculation: Simple Ratio Division$$\text{Score} = \frac{\text{Time Spent}}{\text{Course Duration}}$$ </mark>How the math works: This calculation scales linearly from near 0.0 up to 1.0. If a course is 50 hours long and they only stay for 25 hours, the division yields a score of 0.50. The lower their time compared to the duration, the lower their score.

Category 2: Constant Assignment (Sweet-Spot Learners)When it's used: When a user's spent time matches the course duration or goes up to 30% over <mark>(course_duration_hours <= time_spent_hours <= 1.3 * course_duration_hours)</mark>.The Calculation: Direct Scalar Mapping

$$\text{Score} = 1.0$$

How the math works: There is no division or subtraction here. The model bypasses calculations and directly assigns a flat, perfect score of 1.0. This treats any time spent within this entire baseline range as equally ideal.

Category 3: Linear Penalty with a Hard Floor (Over-Time Learners)When it's used: When a user's spent time exceeds the 30% safety buffer <mark>(time_spent_hours > 1.3 * course_duration_hours)</mark>.The Calculation: Subtractive Distance Penalty combined with a Maximum Boundary Function$$\text{Score} = \max\left(1.0 - \frac{\text{Time Spent} - \text{Buffer Threshold}}{\text{Course Duration}},\, 0.1\right)$$

In [ ]:

# Normalize rating to a 0.1 - 1.0 scale to match the others
df['Normalized_Rating'] = df['rating'] / 5.0

# Calculate a weighted satisfaction index
# Giving 40% weight to explicit rating, 40% to engagement, and 20% to text feedback proxy
df['Value_Score'] = (df['Normalized_Rating'] * 0.4) + (df['Engagement_Index'] * 0.4) + (df['feedback_score'] * 0.2)

df['Value_Score']

In [ ]:
df['Value_Score'].describe()


In [ ]:
# Use np.inf as the final boundary to automatically capture all high-value outliers
df['User_Experience_Tier'] = pd.cut(df['previous_courses_taken'], 
                                    bins=[-1, 2, 6, np.inf], 
                                    labels=[0, 1, 2])

# Now it can safely convert to integer because there are zero unmapped NaN values!
df['User_Experience_Tier'] = df['User_Experience_Tier'].astype(int)

# Let's verify that the value distribution matches our expectations
print(df['User_Experience_Tier'].value_counts())
print("=="*30)
print(df[['previous_courses_taken', 'User_Experience_Tier']].head(10))

In [ ]:
# Calculate the true average engagement for each unique course
course_quality_map = df.groupby('course_id')['Engagement_Index'].mean().to_dict()

# Map it back as a permanent feature of the course
df['Course_Expected_Engagement'] = df['course_id'].map(course_quality_map)

df['Course_Expected_Engagement']

In [ ]:
df['Course_Expected_Engagement'].describe()

In [ ]:
# =====================================================================
# PART 2: ADVANCED FEATURE INTERACTIONS & CROSS-ENGINEERING
# =====================================================================

# 1. Diligence Ratio
df['Diligence_Ratio'] = df['time_spent_hours'] / (df['course_duration_hours'] + 1e-5)

# 2. Price Density per Hour
df['Price_Per_Hour'] = df['course_price'] / (df['course_duration_hours'] + 1e-5)

# --- FIX: Create the difficulty encoding column right here so it exists! ---
difficulty_map = {'Beginner': 0, 'Intermediate': 1, 'Advanced': 2}
df['difficulty_level_enc'] = df['difficulty_level'].map(difficulty_map)

# 3. Structural Difficulty vs Experience Mismatch
df['Difficulty_vs_Experience'] = df['difficulty_level_enc'] - df['User_Experience_Tier']

print("--- Part 2: Advanced Feature Engineering Complete! ---")
print(f"Dataset shape updated to: {df.shape}")
print()
print("Verification Sample of New Interactions:")
print(df[['time_spent_hours', 'course_duration_hours', 'Diligence_Ratio', 'Price_Per_Hour', 'Difficulty_vs_Experience']].head())

In [ ]:
df.columns

## encoding and Tfidf

### Label Encoding for Categorical Features

In [ ]:


# 1. Initialize LabelEncoder for simple binary columns
le = LabelEncoder()
df['certification_offered_enc'] = le.fit_transform(df['certification_offered'])
df['study_material_available_enc'] = le.fit_transform(df['study_material_available'])

# 2. Manual Ordinal Mapping for Difficulty (Ensuring Beginner < Intermediate < Advanced)
difficulty_map = {'Beginner': 0, 'Intermediate': 1, 'Advanced': 2}
df['difficulty_level_enc'] = df['difficulty_level'].map(difficulty_map)

print("Difficulty level mapping verification:")
print(df[['difficulty_level', 'difficulty_level_enc']].drop_duplicates().sort_values('difficulty_level_enc'))

In [ ]:
# 1. Initialize TF-IDF Vectorizer (limiting to top 20 words to avoid overfitting)
tfidf = TfidfVectorizer(max_features=20, stop_words='english')

# 2. Extract vocabulary weight matrices from the textual course name descriptions
tfidf_sparse = tfidf.fit_transform(df['course_name'])
tfidf_df = pd.DataFrame(tfidf_sparse.toarray(), columns=[f"word_{w}" for w in tfidf.get_feature_names_out()])



## Creating the Target Variable


In [ ]:
# Normalizing the Rating

# Map the explicit 1-5 rating into a 0.0 to 1.0 scale
df['Normalized_Rating'] = (df['rating'] - 1) / (5 - 1)

In [ ]:
df.columns.to_list()

In [ ]:
# Defining balance weights
w_rating = 0.40
w_engagement = 0.40
w_feedback = 0.20

# Compute the composite structural utility score
df['Utility_Score'] = (
    (df['Normalized_Rating'] * w_rating) + 
    (df['Engagement_Index'] * w_engagement) + 
    (df['feedback_score'] * w_feedback)
)


In [ ]:
# Find the exact median point of your composite utility score
utility_median = df['Utility_Score'].median()

# Binarize into your clean target feature
df['Recommended'] = (df['Utility_Score'] >= utility_median).astype(int)

# Verify the clean, balanced target distribution!
print('Balanced Target Distribution:')
print(df['Recommended'].value_counts())
print()
print(f"Positive class ratio: {df['Recommended'].mean():.2%}")

In [ ]:
df.head()

## Selecting Features & Scaling

In [ ]:
# =====================================================================
# PART 3 (CONTINUED): MATRIX CONFIGURATION
# =====================================================================
import pandas as pd

# 1. Build the core numeric feature matrix, explicitly retaining our high-impact interactions
X_base = df.drop(columns=[
    # Drop target columns and components used to construct the target formula
    'Recommended', 'Utility_Score', 'Value_Score', 'Normalized_Rating', 
    'rating', 'Engagement_Index', 'feedback_score', 'time_spent_hours', 'course_duration_hours',
    
    # Drop tracking identification metrics and high-cardinality noise
    'user_id', 'course_id', 'instructor',
    
    # Drop raw unencoded text strings
    'course_name', 'certification_offered', 'difficulty_level', 'study_material_available'
]).reset_index(drop=True)

# 2. Horizontally attach your descriptive TF-IDF word columns side-by-side
X = pd.concat([X_base, tfidf_df], axis=1)

# Isolate target array
y = df['Recommended']

print("--- Final Training Matrix Processed! ---")
print(f"Total rows: {X.shape[0]} | Total training features: {X.shape[1]}")
print("\nFinal Column Schema List for Scaling:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i}. {col}")

In [ ]:
df.info()

In [ ]:
# 1. Split the newly updated features matrix
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Re-initialize and completely fit the scaler on the new matrix shape
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print("Data arrays successfully re-shaped and scaled!")

In [ ]:
# Fit on train only — transform both
scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Scaling complete.')
print('Mean of first feature (train, post-scale):', X_train_sc[:, 0].mean().round(6))

## Model Building

### Logistic Regression (Baseline)

In [ ]:
# Logistic Regression as a simple baseline to compare against
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_sc, y_train)

lr_preds = lr_model.predict(X_test_sc)
lr_acc   = accuracy_score(y_test, lr_preds)
lr_auc   = roc_auc_score(y_test, lr_model.predict_proba(X_test_sc)[:, 1])

print(f'Logistic Regression Accuracy : {lr_acc:.4f}')
print(f'Logistic Regression AUC      : {lr_auc:.4f}')
print()
print(classification_report(y_test, lr_preds, target_names=['Not Recommended', 'Recommended']))

### Decision Tree Classifier

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=10, min_samples_split=20, random_state=42)
dt_model.fit(X_train_sc, y_train)

dt_preds = dt_model.predict(X_test_sc)
dt_acc   = accuracy_score(y_test, dt_preds)
dt_auc   = roc_auc_score(y_test, dt_model.predict_proba(X_test_sc)[:, 1])

print(f'Decision Tree Accuracy : {dt_acc:.4f}')
print(f'Decision Tree AUC      : {dt_auc:.4f}')
print()
print(classification_report(y_test, dt_preds, target_names=['Not Recommended', 'Recommended']))

### Random Forest Classifier

In [ ]:
# Random Forest — ensemble of trees, usually performs very well on tabular data
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_sc, y_train)

rf_preds = rf_model.predict(X_test_sc)
rf_acc   = accuracy_score(y_test, rf_preds)
rf_auc   = roc_auc_score(y_test, rf_model.predict_proba(X_test_sc)[:, 1])

print(f'Random Forest Accuracy : {rf_acc:.4f}')
print(f'Random Forest AUC      : {rf_auc:.4f}')
print()
print(classification_report(y_test, rf_preds, target_names=['Not Recommended', 'Recommended']))

### Gradient Boosting Classifier

In [ ]:
gb_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)
gb_model.fit(X_train_sc, y_train)

gb_preds = gb_model.predict(X_test_sc)
gb_acc   = accuracy_score(y_test, gb_preds)
gb_auc   = roc_auc_score(y_test, gb_model.predict_proba(X_test_sc)[:, 1])

print(f'Gradient Boosting Accuracy : {gb_acc:.4f}')
print(f'Gradient Boosting AUC      : {gb_auc:.4f}')
print()
print(classification_report(y_test, gb_preds, target_names=['Not Recommended', 'Recommended']))

In [ ]:
# Initialize LightGBM with robust regularization to optimize scores
lgb_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    random_state=42
)

lgb_model.fit(X_train_sc, y_train)
lgb_preds = lgb_model.predict(X_test_sc)
lgb_acc = accuracy_score(y_test, lgb_preds)

print(f"LightGBM Test Accuracy : {lgb_acc:.4f}")


In [ ]:
from sklearn.model_selection import GridSearchCV


# 1. Define the collection of candidate algorithms
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42)
}

# 2. Define custom hyperparameter grids tailored strictly to each specific model
param_grids = {
    'LogisticRegression': {
        'C': [0.1, 1.0, 10.0]  # Regularization strength
    },
    'RandomForest': {
        'n_estimators': [100, 200],
        'max_depth': [5, 8],
        'min_samples_split': [2, 5]
    },
    'GradientBoosting': {
        'n_estimators': [100, 150],
        'learning_rate': [0.1, 0.2],
        'max_depth': [3, 4]
    }
}

# Dictionary to keep track of the champion versions of each model family
best_models = {}

# 3. Loop through each model family, run its own Grid Search, and find its best score
for model_name in models.keys():
    print(f"=== Starting Grid Search Optimization for: {model_name} ===")
    
    grid_search = GridSearchCV(
        estimator=models[model_name],
        param_grid=param_grids[model_name],
        cv=3,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    
    # Fit the isolated grid on your scaled training metrics
    grid_search.fit(X_train_sc, y_train)
    
    # Store the optimized estimator object
    best_models[model_name] = grid_search.best_estimator_
    
    print(f"Best Params for {model_name}: {grid_search.best_params_}")
    print(f"Best Cross-Validation Score: {grid_search.best_score_:.4f}\n")

print("==================================================")
print("All model optimizations completed successfully! You can now validate them on the test set.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

models_info = [
    ('Logistic Regression', lr_preds),
    ('Decision Tree',       dt_preds),
    ('Random Forest',       rf_preds),
    ('Gradient Boosting',   gb_preds),
]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, (name, preds) in zip(axes, models_info):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    ax.set_title(name, fontsize=13, pad=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# FINAL STEP: RETRIEVAL ENGINE FOR REAL-TIME COURSE RECOMMENDATION
# =====================================================================

def recommend_top_5_courses(user_id_input, main_df, correct_features, my_model, my_scaler, my_tfidf):
    # Quick safety check: does this student actually exist in our records?
    if user_id_input not in main_df['user_id'].values:
        print(f"Error: Could not find any student with User ID {user_id_input}!")
        return
    
    print(f"--- Fetching platform profile and history for Student ID: {user_id_input} ---")
    
    # 1. Pull the static details for this user from their history rows
    user_data = main_df[main_df['user_id'] == user_id_input]
    current_tier = user_data['User_Experience_Tier'].iloc[0]
    total_past_courses = user_data['previous_courses_taken'].iloc[0]
    
    # 2. Get a list of unique courses available on our platform
    # We drop duplicates so we only evaluate each course title once
    course_catalog = main_df.drop_duplicates(subset=['course_name']).copy()
    
    # 3. Filter out courses this student has already taken (we shouldn't recommend them again)
    past_completed_courses = user_data['course_name'].unique()
    available_courses = course_catalog[~course_catalog['course_name'].isin(past_completed_courses)].copy()
    
    # Simple check just in case they finished everything
    if available_courses.empty:
        print(f"Student {user_id_input} has already finished every course in the catalog!")
        return
        
    # 4. Generate our new interaction columns manually for these candidate courses
    # We use 1.0 as a standard baseline behavior setting for Diligence_Ratio
    available_courses['Diligence_Ratio'] = 1.0
    available_courses['Price_Per_Hour'] = available_courses['course_price'] / (available_courses['course_duration_hours'] + 1e-5)
    available_courses['Difficulty_vs_Experience'] = available_courses['difficulty_level_enc'] - current_tier
    
    # 5. Extract the TF-IDF word columns for these catalog names
    text_sparse_matrix = my_tfidf.transform(available_courses['course_name'])
    text_features_df = pd.DataFrame(
        text_sparse_matrix.toarray(), 
        columns=[f"word_{w}" for w in my_tfidf.get_feature_names_out()],
        index=available_courses.index
    )
    
    # 6. Build the inference matrix to match our precise X training layout
    # First get the numeric base columns, then attach the TF-IDF text frames side-by-side
    base_columns_needed = [col for col in X_base.columns]
    inference_X = available_courses[base_columns_needed]
    inference_X = pd.concat([inference_X, text_features_df], axis=1)
    
    # Double check that the column order aligns 100% with our training variables list
    inference_X = inference_X[correct_features]
    
    # 7. Pass through our fitted scaler weights
    inference_X_scaled = my_scaler.transform(inference_X)
    
    # 8. Calculate recommendation match probabilities using our best model
    # Column 1 represents the probability of matching our 'Recommended' target
    scores = my_model.predict_proba(inference_X_scaled)[:, 1]
    available_courses['Match_Score'] = scores
    
    # 9. Sort and select the top 5 highest matching options
    final_top_5 = available_courses.sort_values(by='Match_Score', ascending=False).head(5)
    
    # 10. Print the final recommendation results in a clear format
    print("\n" + "="*65)
    print(f"🎯 TOP 5 PERSONALIZED RECOMMENDATION RESULTS")
    print(f"Student Status: Profile Tier {current_tier} | Courses Completed: {total_past_courses}")
    print("="*65)
    
    rank = 1
    for idx, row in final_top_5.iterrows():
        print(f"{rank}. Course: {row['course_name']}")
        print(f"   Instructor: {row['instructor']} | Difficulty: {row['difficulty_level']}")
        print(f"   Price: ${row['course_price']:.2f} | Match Probability: {row['Match_Score']:.2%}")
        print("-" * 50)
        rank += 1

## Deployment

In [ ]:
import pickle

# 1. Save the trained Gradient Boosting model
with open('model.pkl', 'wb') as f:
    pickle.dump(gb_model, f)

# 2. Save the fitted Standard Scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# 3. Save the fitted TF-IDF Vectorizer
with open('tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("🎉 Success! model.pkl, scaler.pkl, and tfidf.pkl have been created cleanly.")